In [1]:
import pandas as pd
import numpy as np
from glob import glob
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 计算函数
# 计算dafai
def calculate_dafai(L_down, zL_down, Am, Bm):
    """
    根据稳定度条件计算 Dafai 值
    Parameters:
    -----------
    L_down : array_like
        Obukhov长度数组
    zL_down : array_like
        z/L 数组（高度除以 Obukhov 长度）
    Am : float
        不稳定条件下的经验常数
    Bm : float
        稳定条件下的经验常数
    
    Returns:
    --------
    Dafai : ndarray
        计算得到的 Dafai 数组，与输入形状相同
    """
    # 确保输入是 NumPy 数组
    L_down = np.asarray(L_down)
    zL_down = np.asarray(zL_down)
    
    # 创建布尔掩码
    unstable_mask = L_down < 0
    stable_mask = L_down >= 0
    
    # 初始化结果数组
    Dafai = np.full_like(L_down, np.nan)
    
    # 处理不稳定部分 (L_down < 0)
    if np.any(unstable_mask):
        x = (1 - Am * zL_down[unstable_mask]) ** (1/4)
        Dafai[unstable_mask] = (2 * np.log((1 + x) / 2) + 
                                np.log((1 + x**2) / 2) - 
                                2 * np.arctan(x) + 
                                np.pi / 2)
    
    # 处理稳定部分 (L_down >= 0)
    if np.any(stable_mask):
        Dafai[stable_mask] = -Bm * zL_down[stable_mask]
    
    return Dafai

In [3]:
# 读取，重命名文件
files = '/share/home/jinhm/PhD_Project/B7_WAVE_U/CSVData/FullWaveData_202404-202405/平均量/*'
datas = []
for file in sorted(glob(files)):
    data = pd.read_csv(file)

    # 提取文件名
    base_name = file.split('/')[-1].split('.')[0]  # '20240520222'
    # 如果格式固定为 YYYYMMDDHHMM
    year = base_name[0:4]
    month = base_name[4:6]
    day = base_name[6:8]
    hour = base_name[8:10]
    minute = base_name[10:12]

    x = data.to_numpy()[0]
    x = np.insert(x,0,f'{year}{month}{day}{hour}{minute}')
    datas.append(x)
datas = np.array(datas)
df = pd.DataFrame(datas,columns=[
    'Time',
    'depth',
    'U_down',
    'U_upper',
    'u_star_down',
    'u_star_upper',
    'u_star_t_down',
    'u_star_t_upper',
    'f_limit_down_low',
    'f_limit_down_high',
    'angle_u_down',
    'angle_w_down',
    'angle_u_upper',
    'angle_w_upper',
    'fp'
])
df.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/Mean_quantity.xlsx', index=False)

# 生产新列名文件
files_wave = '/share/home/jinhm/PhD_Project/B7_WAVE_U/CSVData/FullWaveData_202404-202405/depth_and_direction_and_Hs.csv'
data = pd.read_csv(files_wave)
data.columns = ['Time', 'depth_wave', 'direction','Hs']
data.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/depth_and_direction_and_Hs.xlsx', index=False)

# 生产新列名文件
files_mean_csv = '/share/home/jinhm/PhD_Project/B7_WAVE_U/CSVData/FullWaveData_202404-202405/mean data.csv'
data = pd.read_csv(files_mean_csv)
data.columns = [
    'Time', 
    'U_down_mean', 
    'Dir_down_mean',
    'U_upper_mean',
    'Dir_upper_mean',
    'Tv_down_mean',
    'Tpotential_down_mean',
    'Tv_upper_mean',
    'Tpotential_upper_mean',
    'wT_down_mean',
    'wT_upper_mean',
    'wTp_down_mean',
    'wTp_upper_mean',
    ]
data.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/Mean_data.xlsx', index=False)

In [ ]:
# 计算 Kp Cp
from scipy.io import loadmat
import os


data = []
files = '/share/home/jinhm/PhD_Project/B7_WAVE_U/CSVData/FullWaveData_202404-202405/波动量/*'
# 构建文件路径
for file in sorted(glob(files)):
    num = pd.read_csv(file).to_numpy()
    file_name = os.path.basename(file)
    # 去掉扩展名
    time_str = os.path.splitext(file_name)[0]

    file_mean_file = f'/share/home/jinhm/PhD_Project/B7_WAVE_U/CSVData/FullWaveData_202404-202405/平均量/{time_str}.csv'

    if not os.path.exists(file_mean_file):
        continue
    U_down = pd.read_csv(file_mean_file).to_numpy()[0,1]

    # 跳过复数
    try:
        U_down = np.float64(U_down)
    except:
        print(U_down)
        continue

    # 提取各列数据（Python索引从0开始）
    f_1Hz2 = num[:, 0]          # 第1列：频率 (Hz)
    Hs_frequency = num[:, 5]     # 第6列：有效波高（由谱积分得到）
    c = num[:, 6]                # 第7列：相速度 (m/s)
    k = num[:, 19]               # 第20列：波数 (rad/m)

    # 寻找相速度最接近风速的频率（下层）
    idx_down = np.argmin(np.abs(c - U_down))
    f_1Hz_plot_down= f_1Hz2[idx_down]

    # # 寻找相速度最接近风速的频率（上层）
    # idx_upper = np.argmin(np.abs(c - U_upper))
    # f_1Hz_plot_upper = f_1Hz2[idx_upper]

    # 寻找谱峰频率（在频率 > 0.09 Hz 的范围内）的 1hz 波速，波数
    mask_high_freq = f_1Hz2 > 0.09

    f_filtered = f_1Hz2[mask_high_freq]
    c_filtered = c[mask_high_freq]
    k_filtered = k[mask_high_freq]

    # 找到Hs最大的索引
    Hs_filtered = Hs_frequency[mask_high_freq]
    idx_peak = np.argmax(Hs_filtered)

    # 谱峰频率和对应的相速度
    fp = f_filtered[idx_peak]
    cp = c_filtered[idx_peak]
    kp = k_filtered[idx_peak]

    data.append([f'{time_str}',fp,cp,kp])

data = np.array(data)
df = pd.DataFrame(data,columns=[
    'Time',
    'fp_comp',
    'cp',
    'kp',
])
df.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/fp_cp_kp.xlsx',index=False)
print('计算完成✅')

In [5]:
# 合并，清洗文件
# 1. 读取三个文件
df_depth = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/depth_and_direction_and_Hs.xlsx')
df_mean = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/Mean_data.xlsx')
df_mean_quantity = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/Mean_quantity.xlsx')
df_fp_cp_kp = pd.read_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/fp_cp_kp.xlsx')


# 2. 统一时间列格式（确保所有时间列都是字符串类型）
df_depth['Time'] = df_depth['Time'].astype(str)
df_mean['Time'] = df_mean['Time'].astype(str)
df_mean_quantity['Time'] = df_mean_quantity['Time'].astype(str)
df_fp_cp_kp['Time'] = df_fp_cp_kp['Time'].astype(str)

# 3. 开始合并 - 逐次合并
# 先合并前两个文件
df_merged = pd.merge(
    df_depth,
    df_mean,
    on='Time',
    how='outer'  # 只保留匹配的行
)

# 再合并第三个文件
df_final = pd.merge(
    df_merged,
    df_mean_quantity,  # 第三个DataFrame
    on='Time',  # 确保第三个文件也有'Time'列
    how='outer'  # 只保留匹配的行
)

# 再合并第四个文件
df = pd.merge(
    df_final,
    df_fp_cp_kp,  # 第三个DataFrame
    on='Time',  # 确保第三个文件也有'Time'列
    how='outer'  # 只保留匹配的行
)

# 删除存在nan的行，和存在复数的行
df_clean = df.dropna()  # 先删缺失
df_clean['u_star_down_real'] = pd.to_numeric(df_clean['u_star_down'], errors='coerce')
df_clean = df_clean.dropna(subset=['u_star_down_real']).drop(columns=['u_star_down_real'])

# 保存结果
df_clean.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/all.xlsx', index=False)

In [6]:
# 计算物理参数
file = '/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/all.xlsx'
data = pd.read_excel(file)

# 物理项常数
k = 0.4
g = 9.8
Am= 19  # Högström (1988)
Bm= 6   # Högström (1988)
v = 1.81e-5  # m²/s：空气运动粘性系数

# 观测数据
depth = data['depth'].to_numpy()

Tv_down_mean = data['Tv_down_mean']
Tv_upper_mean = data['Tv_upper_mean']

wTp_down_mean = data['wTp_down_mean']
wTp_upper_mean = data['wTp_upper_mean']

U_down = data['U_down']
U_upper = data['U_upper']
u_star_down = data['u_star_down']
u_star_upper = data['u_star_upper']

# 计算 z_down,z_upper 202404-202405
# 由于仪器高度调整，深度对应进行调整
depth[depth < 15] += 2

# 计算z_down，z_uppers
z_down = 16+11-depth-6
z_upper = 16+11-depth-2

# 计算 L_down,L_upper
L_down = -np.abs(u_star_down)**3 / k / g / wTp_down_mean * (Tv_down_mean + 273.13)
L_upper = -np.abs(u_star_upper)**3 / k / g / -wTp_upper_mean * (Tv_upper_mean + 273.13)

# 计算大气稳定度z/l, zl_down, zl_upper
zL_down, zL_upper= z_down / L_down , z_upper / L_upper
z10 = np.full_like(u_star_down,10)
z10_down = z10 / L_down

# 计算 dafai
Dafai_down = calculate_dafai(z_down,zL_down,Am,Bm)
Dafai_upper = calculate_dafai(z_upper,zL_upper,Am,Bm)

# 计算 z0
z0 = z_down / np.exp(U_down * k / u_star_down + Dafai_down)

# 计算 charnock ，0.11：光滑流条件下的经验常数
charnock = (z0 - 0.11 * v / u_star_down) * g / u_star_down**2
# 计算 z0_rough
z0_rough = z0 - 0.11 * v / u_star_down

# 矫正风速到中心条件下，10m风速处
u10 = U_down - (u_star_down/k)*np.log(z_down/10)+(u_star_down/k)*Dafai_down
u10_upper = U_upper - (u_star_upper/k)*np.log(z_upper/10)+(u_star_upper/k)*Dafai_upper

data['u10'] = u10
data['u10_upper'] = u10_upper
data['z0'] = z0
data['charnock'] = charnock
data['z0_rough'] = z0_rough
data['z_down'] = z_down
data['z_upper'] = z_upper
# 波陡
data['wave_steepnessdou'] = (data['Hs']*data['kp'])/ 2*np.pi
# 波龄
data['B*'] = data['cp'] / u10
data['z/l'] = zL_down

data['Cdn10'] = (u_star_down / u10) **2

data.to_excel(file,index=False)
print('计算完成✅')

计算完成✅


In [ ]:
# 提取参数话数据集
file = '/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/all.xlsx'
data = pd.read_excel(file)[[
    'Time',
    'U_down',
    'U_upper',
    'u10',
    'u_star_down',
    'u_star_upper',
    'fp_comp',
    'cp',
    'kp',
    'Hs',
    'direction',
    'z0',
    'charnock',
    'z0_rough',
    'wave_steepnessdou',
    'Cdn10',
    'B*',
    'z/l',
    'u10_upper'
]]
data.to_excel('/share/home/jinhm/PhD_Project/B7_WAVE_U/data/202404-202405/DataSet.xlsx',index=False)
print('计算完成✅')